# Laboratorio 10 — Task 2: explicabilidad con SHAP

**Semilla:** 42  
**Modelo:** `GradientBoostingClassifier(n_estimators=200, max_depth=3, random_state=42)`  
**Entorno de la ejecución reportada:** NumPy 2.5.1, pandas 2.3.3, scikit-learn 1.9.0, shap 0.54.1 (los tiempos de 2.4 y 2.6d dependen de la máquina).



## Preparación y fuente de datos

Las 13 variables categóricas se codifican mediante `OrdinalEncoder`, ajustado únicamente con entrenamiento. Sus códigos son **arbitrarios**: tratarlos como números introduce un orden y distancias que no tienen significado semántico. En particular, los árboles pueden crear cortes sobre ese orden artificial; esta es una limitación tanto del modelo como de sus explicaciones.

In [1]:
from __future__ import annotations

import json
import math
import time
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from scipy.io import arff
from scipy.special import expit
from scipy.stats import spearmanr
from sklearn.compose import ColumnTransformer
from sklearn.datasets import fetch_openml
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder

SEED = 42
np.random.seed(SEED)
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "credit-g.arff"
RESULTS_DIR = PROJECT_ROOT / "results" / "task2"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


def scalar_expected_value(explainer) -> float:
    return float(np.asarray(explainer.expected_value).reshape(-1)[0])


def load_credit_g() -> tuple[pd.DataFrame, pd.Series]:
    raw, _ = arff.loadarff(DATA_PATH)
    frame = pd.DataFrame(raw)
    for column in frame.select_dtypes(include="object"):
        frame[column] = frame[column].str.decode("utf-8")
    y = frame.pop("class").map({"good": 1, "bad": 0}).astype(int)
    return frame, y

# Instrucción de descarga indicada en la consigna (la ejecución usa el ARFF local equivalente):
# datos = fetch_openml("credit-g", version=1, as_frame=True)

C:\Users\richi\Documents\2026_S2_Local\Deep-Learning\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Task 2.1 — Preparación, entrenamiento y desempeño

In [2]:
X, y = load_credit_g()
categorical_columns = X.select_dtypes(include="object").columns.tolist()
numeric_columns = X.select_dtypes(exclude="object").columns.tolist()

train_index, test_index = train_test_split(
    X.index, test_size=0.20, stratify=y, random_state=SEED
)
X_train_raw, X_test_raw = X.loc[train_index].copy(), X.loc[test_index].copy()
y_train, y_test = y.loc[train_index].copy(), y.loc[test_index].copy()

encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
X_train = X_train_raw.copy()
X_test = X_test_raw.copy()
X_train[categorical_columns] = encoder.fit_transform(X_train_raw[categorical_columns])
X_test[categorical_columns] = encoder.transform(X_test_raw[categorical_columns])
X_train = X_train.astype(float)
X_test = X_test.astype(float)

model = GradientBoostingClassifier(
    n_estimators=200, max_depth=3, random_state=SEED
)
model.fit(X_train, y_train)
test_probability = model.predict_proba(X_test)[:, 1]
test_prediction = (test_probability >= 0.5).astype(int)
accuracy = accuracy_score(y_test, test_prediction)
auc = roc_auc_score(y_test, test_probability)

display(pd.DataFrame([{'accuracy': accuracy, 'auc': auc, 'train_rows': len(X_train), 'test_rows': len(X_test)}]).style.format(precision=6))

,accuracy,auc,train_rows,test_rows
0,0.775000,0.765000,800,200


### Respuesta 2.1

La división estratificada produjo **800 solicitudes de entrenamiento** y **200 de prueba**, manteniendo la proporción total de 700 casos `good` y 300 `bad`. En prueba se obtuvo:

| Métrica | Resultado |
|---|---:|
| Exactitud | **0.7750** |
| AUC ROC | **0.7650** |

La exactitud indica que el modelo acertó el 77.5% de las decisiones con umbral 0.5. El AUC de 0.765 mide una capacidad de ordenamiento moderada: una solicitud `good` elegida al azar recibe mayor puntuación que una `bad` aproximadamente el 76.5% de las veces. Estas cifras deben interpretarse junto con la limitación del `OrdinalEncoder`, porque los cortes de los árboles dependen del orden numérico arbitrario asignado a las categorías.

## Task 2.2 — TreeSHAP intervencional

In [3]:
rng = np.random.default_rng(SEED)
background_index = rng.choice(X_train.index.to_numpy(), size=100, replace=False)
background = X_train.loc[background_index]

tree_explainer = shap.TreeExplainer(
    model,
    data=background,
    feature_perturbation="interventional",
    model_output="raw",
)
tree_explanation = tree_explainer(X_test, check_additivity=True)
tree_values = np.asarray(tree_explanation.values)
base_value = scalar_expected_value(tree_explainer)
raw_test = model.decision_function(X_test)
efficiency_error = float(np.max(np.abs(base_value + tree_values.sum(axis=1) - raw_test)))
background_raw_mean = float(model.decision_function(background).mean())
base_mean_error = abs(base_value - background_raw_mean)

global_importance = np.abs(tree_values).mean(axis=0)
permutation = permutation_importance(
    model,
    X_test,
    y_test,
    scoring="roc_auc",
    n_repeats=10,
    random_state=SEED,
)
importance_df = pd.DataFrame(
    {
        "feature": X.columns,
        "mean_abs_shap": global_importance,
        "permutation_auc_decrease": permutation.importances_mean,
        "permutation_std": permutation.importances_std,
    }
)
importance_df["shap_rank"] = importance_df["mean_abs_shap"].rank(
    ascending=False, method="min"
).astype(int)
importance_df["permutation_rank"] = importance_df[
    "permutation_auc_decrease"
].rank(ascending=False, method="min").astype(int)
importance_df["rank_difference"] = (
    importance_df["shap_rank"] - importance_df["permutation_rank"]
).abs()
ranking_spearman = float(
    spearmanr(importance_df["shap_rank"], importance_df["permutation_rank"]).statistic
)
importance_df.to_csv(RESULTS_DIR / "task2_2c_importances.csv", index=False)

approved_position = int(np.argmax(test_probability))
rejected_position = int(np.argmin(test_probability))
selected_rows = []
for label, position in [("approved", approved_position), ("rejected", rejected_position)]:
    order = np.argsort(np.abs(tree_values[position]))[::-1][:3]
    for rank, feature_position in enumerate(order, start=1):
        feature = X.columns[feature_position]
        selected_rows.append(
            {
                "client": label,
                "test_position": position,
                "original_index": int(X_test.index[position]),
                "probability_good": test_probability[position],
                "rank": rank,
                "feature": feature,
                "encoded_value": X_test.iloc[position, feature_position],
                "original_value": str(X_test_raw.iloc[position][feature]),
                "shap_log_odds": tree_values[position, feature_position],
            }
        )
selected_df = pd.DataFrame(selected_rows)
selected_df.to_csv(RESULTS_DIR / "task2_2d_selected_clients.csv", index=False)

for label, position in [("approved", approved_position), ("rejected", rejected_position)]:
    plt.figure()
    shap.plots.waterfall(tree_explanation[position], max_display=10, show=False)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / f"task2_2d_waterfall_{label}.png", dpi=180, bbox_inches="tight")
    plt.close()

plt.figure()
shap.plots.beeswarm(tree_explanation, max_display=15, show=False)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "task2_2d_beeswarm.png", dpi=180, bbox_inches="tight")
plt.close()

print(f'Error máximo de eficiencia: {efficiency_error:.12e}')
print(f'Valor base: {base_value:.12f}')
print(f'Promedio log-odds del fondo: {background_raw_mean:.12f}')
print(f'Error valor base-promedio: {base_mean_error:.12e}')
display(importance_df.sort_values('shap_rank').head(10).style.format(precision=6))
display(selected_df.style.format({'probability_good': '{:.6f}', 'shap_log_odds': '{:+.6f}'}))

Error máximo de eficiencia: 9.701241499016e-08
Valor base: 1.430742147497
Promedio log-odds del fondo: 1.430742147497
Error valor base-promedio: 0.000000000000e+00


,feature,mean_abs_shap,permutation_auc_decrease,permutation_std,shap_rank,permutation_rank,rank_difference
0,checking_status,0.760304,0.119000,0.030147,1,1,0
3,purpose,0.511731,0.003750,0.010658,2,12,10
4,credit_amount,0.490710,0.076405,0.013491,3,2,1
1,duration,0.391745,0.057905,0.011939,4,3,1
5,savings_status,0.296328,0.026155,0.011215,5,5,0
12,age,0.288582,-0.005417,0.009607,6,20,14
2,credit_history,0.227800,0.039643,0.011225,7,4,3
10,residence_since,0.207841,0.002202,0.013392,8,13,5
7,installment_commitment,0.197878,0.010083,0.004868,9,7,2
13,other_payment_plans,0.174898,0.020119,0.003393,10,6,4


,client,test_position,original_index,probability_good,rank,feature,encoded_value,original_value,shap_log_odds
0,approved,50,175,0.999447,1,purpose,9.000000,used car,+1.463294
1,approved,50,175,0.999447,2,age,53.000000,53.0,+1.104494
2,approved,50,175,0.999447,3,checking_status,3.000000,no checking,+1.006869
3,rejected,76,721,0.051742,1,credit_history,0.000000,all paid,-1.316188
4,rejected,76,721,0.051742,2,savings_status,3.000000,>=1000,+0.972107
5,rejected,76,721,0.051742,3,credit_amount,433.000000,433.0,-0.964673


### Respuestas 2.2a y 2.2b — Eficiencia y valor base

Para las 200 instancias de prueba, el máximo de

$$\left|\phi_0+\sum_j\phi_j-f(x)\right|$$

fue **$9.70\times10^{-8}$**. El valor base fue **1.430742147497 log-odds** y el promedio de la salida del modelo sobre las 100 instancias del fondo fue exactamente **1.430742147497**, con diferencia numérica **0**. Los resultados confirman tanto la eficiencia de los valores SHAP como la interpretación del valor base para el explicador intervencional utilizado.

### Respuesta 2.2c — Importancia global y por permutación

| Posición | Top por $I_j=\operatorname{mean}(|\phi_j|)$ | $I_j$ | Top por permutación | Caída media de AUC |
|---:|---|---:|---|---:|
| 1 | checking_status | 0.760304 | checking_status | 0.119000 |
| 2 | purpose | 0.511731 | credit_amount | 0.076405 |
| 3 | credit_amount | 0.490710 | duration | 0.057905 |
| 4 | duration | 0.391745 | credit_history | 0.039643 |
| 5 | savings_status | 0.296328 | savings_status | 0.026155 |

La correlación de Spearman entre los 20 puestos fue **$\rho=0.6451$**. Existe una asociación positiva moderada entre los rankings, pero no equivalencia: SHAP resume el tamaño medio de las contribuciones a los log-odds, mientras que la permutación mide cuánto empeora el AUC al destruir la información de una variable en prueba. `purpose`, por ejemplo, ocupa el segundo lugar por SHAP pero no aparece entre las cinco primeras por permutación.

### Respuesta 2.2d — Clientes y visualizaciones

Se eligieron los casos más extremos que satisfacen los umbrales solicitados:

- **Cliente aprobado**, índice original 175, $p(	ext{good})=0.999447$. Sus contribuciones de mayor magnitud fueron `purpose = used car` (**+1.4633** log-odds), `age = 53` (**+1.1045**) y `checking_status = no checking` (**+1.0069**). Las tres empujaron la predicción hacia `good` respecto del valor base.
- **Cliente rechazado**, índice original 721, $p(	ext{good})=0.051742$. Las contribuciones principales fueron `credit_history = all paid` (**−1.3162**), `savings_status = >=1000` (**+0.9721**) y `credit_amount = 433` (**−0.9647**). El ahorro amortiguó el rechazo, pero las dos contribuciones negativas de mayor magnitud —junto con el resto— dominaron la decisión. Estas asociaciones pertenecen al modelo y no deben leerse como efectos causales; además, las categorías están codificadas ordinalmente.

![Waterfall del cliente aprobado](../results/task2/task2_2d_waterfall_approved.png)

![Waterfall del cliente rechazado](../results/task2/task2_2d_waterfall_rejected.png)

![Beeswarm global](../results/task2/task2_2d_beeswarm.png)

## Task 2.3 — Variable redundante

In [4]:
# Task 2.3: duplicate the most important numeric feature.
numeric_positions = [X.columns.get_loc(column) for column in numeric_columns]
top_numeric_position = numeric_positions[int(np.argmax(global_importance[numeric_positions]))]
top_numeric_feature = X.columns[top_numeric_position]
duplicate_feature = f"{top_numeric_feature}_dup"

X_encoded_full = pd.concat([X_train, X_test]).loc[X.index].copy()
numeric_sigma = float(X_encoded_full[top_numeric_feature].std(ddof=1))
dup_rng = np.random.default_rng(SEED)
X_encoded_full[duplicate_feature] = (
    X_encoded_full[top_numeric_feature]
    + 0.05 * numeric_sigma * dup_rng.normal(size=len(X_encoded_full))
)
X_train_dup = X_encoded_full.loc[train_index]
X_test_dup = X_encoded_full.loc[test_index]
background_dup = X_train_dup.loc[background_index]

model_dup = GradientBoostingClassifier(
    n_estimators=200, max_depth=3, random_state=SEED
)
model_dup.fit(X_train_dup, y_train)

dup_interventional = shap.TreeExplainer(
    model_dup,
    data=background_dup,
    feature_perturbation="interventional",
    model_output="raw",
)(X_test_dup, check_additivity=True)
dup_path = shap.TreeExplainer(
    model_dup,
    feature_perturbation="tree_path_dependent",
    model_output="raw",
)(X_test_dup, check_additivity=True)

original_dup_position = X_train_dup.columns.get_loc(top_numeric_feature)
copy_dup_position = X_train_dup.columns.get_loc(duplicate_feature)
duplicate_rows = []
for configuration, explanation in [
    ("interventional", dup_interventional),
    ("tree_path_dependent", dup_path),
]:
    values = np.asarray(explanation.values)
    original_values = values[:, original_dup_position]
    copy_values = values[:, copy_dup_position]
    duplicate_rows.append(
        {
            "configuration": configuration,
            "original_feature": top_numeric_feature,
            "mean_abs_phi_original": np.abs(original_values).mean(),
            "mean_abs_phi_copy": np.abs(copy_values).mean(),
            "mean_abs_phi_sum": np.abs(original_values + copy_values).mean(),
            "baseline_mean_abs_phi_original": global_importance[top_numeric_position],
        }
    )
duplicate_df = pd.DataFrame(duplicate_rows)
duplicate_df.to_csv(RESULTS_DIR / "task2_3_duplicate_credit.csv", index=False)

display(duplicate_df.style.format(precision=6))

,configuration,original_feature,mean_abs_phi_original,mean_abs_phi_copy,mean_abs_phi_sum,baseline_mean_abs_phi_original
0,interventional,credit_amount,0.518015,0.428365,0.545704,0.490710
1,tree_path_dependent,credit_amount,0.381243,0.362945,0.530267,0.490710


### Respuesta 2.3

La variable numérica con mayor $I_j$ fue `credit_amount`, con **0.490710** en el modelo original. Su copia se construyó como `credit_amount_dup = credit_amount + 0.05 σ ε`, usando $\sigma=2822.736876$.

| Configuración | Media $|\phi_{orig}|$ | Media $|\phi_{copia}|$ | Media $|\phi_{orig}+\phi_{copia}|$ | $I_j$ original sin copia |
|---|---:|---:|---:|---:|
| Intervencional | 0.518015 | 0.428365 | 0.545704 | 0.490710 |
| Tree path dependent | 0.381243 | 0.362945 | 0.530267 | 0.490710 |

El crédito se reparte entre ambas variables, pero no de forma idéntica. La suma agrupada conserva una magnitud parecida —aunque algo mayor— que la importancia original. La configuración dependiente de trayectorias produjo un reparto más parejo (0.381 frente a 0.363), mientras que la intervencional asignó más a la original (0.518 frente a 0.428).

## Task 2.4 — KernelSHAP frente a TreeSHAP

In [5]:
# Task 2.4: KernelSHAP versus TreeSHAP on ten test instances.
kernel_positions = np.arange(10)
X_kernel = X_test.iloc[kernel_positions]
kernel_rows = []
background_order = np.random.default_rng(SEED + 1).permutation(X_train.index.to_numpy())

def raw_model(values):
    values_frame = pd.DataFrame(values, columns=X.columns)
    return model.decision_function(values_frame)


for background_size in [10, 50, 100]:
    kernel_background = X_train.loc[background_order[:background_size]]
    matching_tree = shap.TreeExplainer(
        model,
        data=kernel_background,
        feature_perturbation="interventional",
        model_output="raw",
    )(X_kernel, check_additivity=True)
    kernel_explainer = shap.KernelExplainer(raw_model, kernel_background)
    start = time.perf_counter()
    kernel_values = np.asarray(
        kernel_explainer.shap_values(X_kernel, nsamples=2000, silent=True)
    )
    elapsed = time.perf_counter() - start
    mae = float(np.mean(np.abs(kernel_values - np.asarray(matching_tree.values))))
    kernel_rows.append(
        {
            "background_size": background_size,
            "instances": len(X_kernel),
            "nsamples_per_instance": 2000,
            "kernel_seconds": elapsed,
            "seconds_per_instance": elapsed / len(X_kernel),
            "mae_vs_matching_tree_shap": mae,
            "model_rows_nominal": background_size * 2000 * len(X_kernel),
        }
    )
kernel_df = pd.DataFrame(kernel_rows)
kernel_df.to_csv(RESULTS_DIR / "task2_4_kernel_vs_tree.csv", index=False)

display(kernel_df.style.format(precision=6))

,background_size,instances,nsamples_per_instance,kernel_seconds,seconds_per_instance,mae_vs_matching_tree_shap,model_rows_nominal
0,10,10,2000,0.805241,0.080524,0.045460,200000
1,50,10,2000,2.644848,0.264485,0.040343,1000000
2,100,10,2000,5.143639,0.514364,0.040586,2000000


### Respuesta 2.4

Se explicaron las mismas 10 instancias, con 2,000 muestras de KernelSHAP por instancia.

| Fondo | Tiempo total (s) | Tiempo por instancia (s) | MAE frente a TreeSHAP |
|---:|---:|---:|---:|
| 10 | 0.8052 | 0.0805 | 0.045460 |
| 50 | 2.6448 | 0.2645 | 0.040343 |
| 100 | 5.1436 | 0.5144 | 0.040586 |

Al aumentar el fondo de 10 a 100, el tiempo creció de 0.81 a 5.14 segundos. El error bajó de 0.0455 a 0.0403 al pasar de 10 a 50 instancias de fondo, y con 100 quedó prácticamente igual (0.0406): la mejora se satura porque, con 2,000 muestras fijas, el ruido del muestreo de coaliciones domina sobre el del fondo. KernelSHAP es aproximado y considerablemente más costoso que TreeSHAP para este modelo de árboles.

## Task 2.5 — Fórmula exacta de Shapley

In [6]:
# Task 2.5: exact Shapley enumeration for six numerical variables.
six_features = [
    "duration",
    "credit_amount",
    "installment_commitment",
    "residence_since",
    "age",
    "existing_credits",
]
X6_train = X_train[six_features]
X6_test = X_test[six_features]
model6 = GradientBoostingClassifier(
    n_estimators=200, max_depth=3, random_state=SEED
)
model6.fit(X6_train, y_train)
background6_index = np.random.default_rng(SEED + 2).choice(
    X6_train.index.to_numpy(), size=50, replace=False
)
background6 = X6_train.loc[background6_index]
X6_eval = X6_test.iloc[:5]


def coalition_value(instance: np.ndarray, coalition: frozenset[int]) -> float:
    hybrid = background6.to_numpy(copy=True)
    for feature_position in coalition:
        hybrid[:, feature_position] = instance[feature_position]
    hybrid_frame = pd.DataFrame(hybrid, columns=six_features)
    return float(model6.decision_function(hybrid_frame).mean())


exact_values = np.zeros((len(X6_eval), len(six_features)))
exact_efficiency_errors = []
coalition_tables = []
M = len(six_features)
for instance_position, instance in enumerate(X6_eval.to_numpy()):
    v = {}
    for coalition_size in range(M + 1):
        for members in combinations(range(M), coalition_size):
            coalition = frozenset(members)
            v[coalition] = coalition_value(instance, coalition)
    coalition_tables.append(v)
    for feature_position in range(M):
        others = [j for j in range(M) if j != feature_position]
        phi = 0.0
        for coalition_size in range(M):
            weight = (
                math.factorial(coalition_size)
                * math.factorial(M - coalition_size - 1)
                / math.factorial(M)
            )
            for members in combinations(others, coalition_size):
                coalition = frozenset(members)
                phi += weight * (
                    v[coalition | {feature_position}] - v[coalition]
                )
        exact_values[instance_position, feature_position] = phi
    exact_efficiency_errors.append(
        abs(
            exact_values[instance_position].sum()
            - (v[frozenset(range(M))] - v[frozenset()])
        )
    )

tree6_explanation = shap.TreeExplainer(
    model6,
    data=background6,
    feature_perturbation="interventional",
    model_output="raw",
)(X6_eval, check_additivity=True)
exact_vs_tree_max_error = float(
    np.max(np.abs(exact_values - np.asarray(tree6_explanation.values)))
)
exact_efficiency_max_error = float(max(exact_efficiency_errors))

exact_rows = []
for instance_position in range(len(X6_eval)):
    for feature_position, feature in enumerate(six_features):
        exact_rows.append(
            {
                "instance": instance_position,
                "feature": feature,
                "exact_phi": exact_values[instance_position, feature_position],
                "tree_phi": np.asarray(tree6_explanation.values)[
                    instance_position, feature_position
                ],
                "abs_error": abs(
                    exact_values[instance_position, feature_position]
                    - np.asarray(tree6_explanation.values)[
                        instance_position, feature_position
                    ]
                ),
            }
        )
pd.DataFrame(exact_rows).to_csv(
    RESULTS_DIR / "task2_5_exact_vs_tree.csv", index=False
)

print(f'Coaliciones por instancia: {2**M}')
print(f'Error máximo frente a TreeSHAP: {exact_vs_tree_max_error:.12e}')
print(f'Error máximo de eficiencia: {exact_efficiency_max_error:.12e}')
display(pd.DataFrame(exact_rows).head(12).style.format(precision=10))

Coaliciones por instancia: 64
Error máximo frente a TreeSHAP: 2.089589545839e-08
Error máximo de eficiencia: 2.498001805407e-16


,instance,feature,exact_phi,tree_phi,abs_error
0,0,duration,-0.4590995504,-0.4590995509,0.0000000005
1,0,credit_amount,0.5067760376,0.5067760343,0.0000000033
2,0,installment_commitment,-0.0357029344,-0.0357029352,0.0000000007
3,0,residence_since,-0.1075489372,-0.1075489386,0.0000000014
4,0,age,0.2882800781,0.2882800765,0.0000000016
5,0,existing_credits,0.1535911764,0.1535911779,0.0000000015
6,1,duration,-1.1754916959,-1.1754917020,0.0000000061
7,1,credit_amount,0.2016050614,0.2016050490,0.0000000124
8,1,installment_commitment,-0.0184577764,-0.0184577764,0.0000000000
9,1,residence_since,-0.0806056729,-0.0806056743,0.0000000014


### Respuesta 2.5

Para cada una de las 5 instancias se enumeraron las **64 coaliciones** de las seis variables y se aplicaron los pesos factoriales de Shapley. El error absoluto máximo frente a TreeSHAP intervencional fue **$2.09\times10^{-8}$**. Además,

$$\sum_i\phi_i=v(N)-v(\varnothing)$$

se cumplió con error máximo **$2.50\times10^{-16}$**. La implementación directa y TreeSHAP coinciden hasta precisión numérica, y la fórmula escrita sin `shap` satisface eficiencia.

## Task 2.6 — Interpretación crítica

In [7]:
# Task 2.6a: non-additivity after the logistic transformation.
rejected_phi = tree_values[rejected_position]
rejected_probability = float(test_probability[rejected_position])
base_probability = float(expit(base_value))
individual_probability_effects = expit(base_value + rejected_phi) - base_probability
sum_individual_probability_effects = float(individual_probability_effects.sum())
joint_probability_change = rejected_probability - base_probability

display(pd.DataFrame([{'p_rejected': rejected_probability, 'base_probability': base_probability, 'sum_individual_effects': sum_individual_probability_effects, 'joint_probability_change': joint_probability_change, 'gap': sum_individual_probability_effects-joint_probability_change}]).style.format(precision=6))

,p_rejected,base_probability,sum_individual_effects,joint_probability_change,gap
0,0.051742,0.807017,-0.945719,-0.755275,-0.190444


### Respuesta 2.6a — Log-odds frente a probabilidad

Para el cliente rechazado, $p(x)=0.051742$ y la probabilidad base es $\sigma(\phi_0)=0.807017$. La suma de los efectos individuales transformados a probabilidad fue **−0.945719**, mientras que el cambio conjunto real fue

$$p(x)-\sigma(\phi_0)=\mathbf{-0.755275}.$$

La diferencia entre ambos resultados es **−0.190444**. SHAP es aditivo en el espacio explicado, aquí log-odds: $f(x)=\phi_0+\sum_j\phi_j$. La función logística es no lineal, por lo que en general $\sum_j[\sigma(\phi_0+\phi_j)-\sigma(\phi_0)]$ no equivale a $\sigma(\phi_0+\sum_j\phi_j)-\sigma(\phi_0)$. Cada efecto individual parte del mismo valor base y no incorpora cómo cambia la pendiente de la sigmoide tras añadir las demás contribuciones.

In [8]:
# Task 2.6b: largest ranking disagreement and correlations.
disagreement_row = importance_df.sort_values(
    ["rank_difference", "feature"], ascending=[False, True]
).iloc[0]
disagreement_feature = disagreement_row["feature"]
full_encoded = pd.concat([X_train, X_test]).loc[X.index]
correlations = (
    full_encoded.corr(method="spearman")[disagreement_feature]
    .drop(disagreement_feature)
    .sort_values(key=np.abs, ascending=False)
)
correlation_df = correlations.rename("spearman_correlation").reset_index(
    name="spearman_correlation"
).rename(columns={"index": "feature"})
correlation_df.to_csv(RESULTS_DIR / "task2_6b_correlations.csv", index=False)

display(importance_df.loc[importance_df.feature.eq(disagreement_feature)].style.format(precision=6))
display(correlation_df.head(10).style.format(precision=6))

,feature,mean_abs_shap,permutation_auc_decrease,permutation_std,shap_rank,permutation_rank,rank_difference
12,age,0.288582,-0.005417,0.009607,6,20,14


,feature,spearman_correlation
0,housing,-0.329090
1,employment,0.280598
2,personal_status,0.255646
3,residence_since,0.234709
4,num_dependents,0.190651
5,credit_history,-0.180716
6,own_telephone,0.178423
7,existing_credits,0.141287
8,checking_status,0.109891
9,property_magnitude,0.108114


### Respuesta 2.6b — Mayor desacuerdo entre rankings

La mayor diferencia correspondió a `age`: puesto **6** por SHAP ($I_j=0.288582$) y puesto **20** por permutación (caída media de AUC **−0.005417 ± 0.009607**), una separación de 14 posiciones. Sus correlaciones de Spearman de mayor magnitud fueron con `housing` (**−0.3291**), `employment` (**0.2806**), `personal_status` (**0.2556**), `residence_since` (**0.2347**) y `num_dependents` (**0.1907**).

Una hipótesis compatible con estos datos es que `age` genera contribuciones locales apreciables en los log-odds, pero parte de su información es redundante o interactúa con vivienda, empleo, estado personal y residencia. Al barajarla, el modelo todavía conserva señales relacionadas y el ordenamiento AUC no empeora; la estimación incluso es ligeramente negativa dentro de su variabilidad. La permutación mide pérdida de desempeño predictivo al destruir una variable, mientras que $I_j$ mide el tamaño medio de la contribución asignada por SHAP dentro del modelo.

### Respuesta 2.6c — Reparto de crédito entre variables redundantes

El axioma de **simetría** establece que dos jugadores con la misma contribución marginal para toda coalición deben recibir el mismo valor de Shapley. Aquí no se cumple su premisa: la copia tiene ruido, el modelo se reentrena y los árboles pueden escoger cortes distintos o usar una variable en trayectorias diferentes. Por eso no existe obligación de repartir el crédito exactamente por mitad.

En la versión intervencional, `credit_amount` recibió 0.518015 y la copia 0.428365 de importancia media absoluta. La variante `tree_path_dependent` repartió 0.381243 y 0.362945, respectivamente. Esta última usa frecuencias de entrenamiento a lo largo de las ramas como sustituto de la distribución de variables ausentes; no calcula la distribución condicional real ni modela explícitamente todas las dependencias. Por ello es solo una aproximación dependiente de la estructura aprendida.

El resultado advierte que una importancia SHAP describe cómo **este modelo**, con **esta definición de ausencia y fondo**, distribuye su predicción. No garantiza la importancia causal o económica de una variable en el mundo real; con predictores redundantes, el crédito puede dividirse o desplazarse sin que cambie mucho la información disponible.

In [9]:
# Task 2.6d: prediction timing and extrapolations.
timing_frame = X_test.iloc[:100]
timing_repetitions = 1000
start = time.perf_counter()
for _ in range(timing_repetitions):
    model.predict_proba(timing_frame)
prediction_seconds_100 = (time.perf_counter() - start) / timing_repetitions
exact_20_seconds = prediction_seconds_100 * (2**20)
exact_60_years_at_microsecond = (2**60 * 1e-6) / (365.25 * 24 * 3600)

summary = {
    "versions": {
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "sklearn": __import__("sklearn").__version__,
        "shap": shap.__version__,
    },
    "dataset": {
        "rows": len(X),
        "features": X.shape[1],
        "numeric_features": len(numeric_columns),
        "categorical_features": len(categorical_columns),
        "train_rows": len(X_train),
        "test_rows": len(X_test),
        "good_total": int(y.sum()),
        "bad_total": int((1 - y).sum()),
    },
    "task_2_1": {"accuracy": accuracy, "auc": auc},
    "task_2_2": {
        "efficiency_max_abs_error": efficiency_error,
        "base_value": base_value,
        "background_raw_mean": background_raw_mean,
        "base_mean_abs_error": base_mean_error,
        "ranking_spearman": ranking_spearman,
        "top5_shap": importance_df.nlargest(5, "mean_abs_shap")[
            ["feature", "mean_abs_shap", "shap_rank"]
        ].to_dict(orient="records"),
        "top5_permutation": importance_df.nlargest(
            5, "permutation_auc_decrease"
        )[["feature", "permutation_auc_decrease", "permutation_rank"]].to_dict(
            orient="records"
        ),
        "approved_position": approved_position,
        "approved_index": int(X_test.index[approved_position]),
        "approved_probability": float(test_probability[approved_position]),
        "rejected_position": rejected_position,
        "rejected_index": int(X_test.index[rejected_position]),
        "rejected_probability": rejected_probability,
        "selected_contributions": selected_rows,
    },
    "task_2_3": {
        "duplicated_feature": top_numeric_feature,
        "duplicate_name": duplicate_feature,
        "sigma": numeric_sigma,
        "rows": duplicate_rows,
    },
    "task_2_4": kernel_rows,
    "task_2_5": {
        "max_abs_error_exact_vs_tree": exact_vs_tree_max_error,
        "max_efficiency_error": exact_efficiency_max_error,
        "instances": len(X6_eval),
        "coalitions_per_instance": 2**M,
    },
    "task_2_6": {
        "rejected_probability": rejected_probability,
        "base_probability": base_probability,
        "sum_individual_probability_effects": sum_individual_probability_effects,
        "joint_probability_change": joint_probability_change,
        "probability_additivity_gap": sum_individual_probability_effects
        - joint_probability_change,
        "disagreement_feature": disagreement_feature,
        "disagreement_shap_rank": int(disagreement_row["shap_rank"]),
        "disagreement_permutation_rank": int(disagreement_row["permutation_rank"]),
        "disagreement_rank_difference": int(disagreement_row["rank_difference"]),
        "top_correlations": correlation_df.head(5).to_dict(orient="records"),
        "prediction_seconds_100": prediction_seconds_100,
        "exact_20_seconds": exact_20_seconds,
        "exact_20_hours": exact_20_seconds / 3600,
        "kernel_seconds_per_instance_background_100": kernel_rows[-1][
            "seconds_per_instance"
        ],
        "exact_60_years_at_one_microsecond_per_value": exact_60_years_at_microsecond,
    },
}

(RESULTS_DIR / "summary.json").write_text(
    json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8"
)
display(pd.DataFrame(kernel_rows).style.format(precision=6))

,background_size,instances,nsamples_per_instance,kernel_seconds,seconds_per_instance,mae_vs_matching_tree_shap,model_rows_nominal
0,10,10,2000,0.805241,0.080524,0.045460,200000
1,50,10,2000,2.644848,0.264485,0.040343,1000000
2,100,10,2000,5.143639,0.514364,0.040586,2000000


### Respuesta 2.6d — Costo computacional

Con 2,000 muestras por instancia, el número nominal de filas evaluadas es proporcional al tamaño del fondo. Al pasar de 10 a 50 observaciones de fondo, el trabajo nominal se multiplicó por 5 y el tiempo por **3.28**; al pasar de 10 a 100, el trabajo se multiplicó por 10 y el tiempo por **6.39**. El crecimiento observado es consistente con la tendencia propuesta, aunque no perfectamente lineal por costos fijos, vectorización y paralelismo. El error bajó de 0.045460 a 0.040343 al pasar de 10 a 50, pero con 100 quedó en 0.040586, sin mejora adicional: más allá de unas 50 instancias de fondo, el error lo domina el muestreo de 2,000 coaliciones y no el tamaño del fondo.

Una predicción sobre 100 instancias tomó en promedio **0.001161 s**. Reutilizando ese tiempo para cada evaluación de $v(S)$, las $2^{20}=1,048,576$ coaliciones requerirían aproximadamente **1,217.0 s**, es decir, **20.3 minutos o 0.338 horas** por instancia. KernelSHAP con fondo 100 tomó **0.5144 s por instancia**, por lo que la estimación exacta sería unas **2,366 veces más lenta**.

Con 60 variables existirían $2^{60}$ coaliciones. Aun suponiendo solo un microsegundo por evaluación, el cálculo tardaría aproximadamente **36,534 años**. Esto ilustra por qué se necesitan algoritmos exactos especializados, como TreeSHAP, o aproximaciones como KernelSHAP.